https://colab.research.google.com/drive/1XGQzOkZLs_IRt_epBiZBiN_rVOFgpMQC#scrollTo=jGbfGrIPCoAT

In [ ]:
%load_ext sql
%sql sqlite:///b02_beyond_the_code.db

# B02 - Beyond-the-Code.ipynb
## Project: Beyond the Code: Who Developers Are and What Drives Them
**Team:** B02  
**Members:** Abhi Jindal, Adam, Daaksha B Arun, Khoi, Jesse

**Notebook contents (outline):**
- Executive summary
- Problem definition & dataset descriptions
- Data loading instructions (SQL-based)
- Cleaning (SQL)
- Exploratory questions + SQL answers
- Mini-conclusions after sections
- Conclusion & recommendations
- References & Generative AI disclosure


## Executive Summary

This notebook analyzes the Stack Overflow Annual Developer Survey (public dataset) joined with McKinsey "State of AI" scraped results (2020–2025). Our goal is to identify:
- Most-used programming languages and tools,
- AI adoption trends among developers,
- Skills in demand and gaps between tooling interest and adoption,
- How McKinsey industry-level AI adoption compares to developer-reported usage.

Key recommendations (preview): focus training on high-growth AI frameworks and cloud tooling; emphasize language + AI combinations employers seek. Outputs are produced using SQL queries (no Python used for data transformations).


## Problem Definition & Data Description

**Problem:** As AI reshapes the job market, which developer skills, languages, and AI tools are growing in demand? Which skills should students prioritize?

**Data:**
- `survey_results_public.csv` — Stack Overflow Annual Developer Survey (public). Key columns typically include `Respondent`, `LanguageWorkedWith`, `LanguageDesireNextYear`, `DevType`, `Country`, `YearsCode`, `Employment`, `Salary`, etc.
- `the-state-of-ai-2023-data.csv` — Scraped McKinsey "State of AI" measures for 2020-2025 (metrics like `year`, `industry_adoption_rate`, `investment_trend`, `skills_gap_indicators`).

**Source URLs:**
- Stack Overflow: https://survey.stackoverflow.co  
- McKinsey: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai


## Data loading (SQL-first approach)

**Approach:** We'll create a local SQLite database `b02_beyond_the_code.db` and import the CSVs using the sqlite3 CLI (so subsequent analysis is purely SQL). If you use Jupyter, prefix these bash lines with `!` to run them in the notebook kernel.

**Steps (run in a code cell):**
1. Create database and import CSVs using sqlite3 CLI. Example commands (run in a notebook `Code` cell with `!` or terminal):

```bash
# from the notebook (prefix each line with ! if running in a Jupyter code cell)
!rm -f b02_beyond_the_code.db
!sqlite3 b02_beyond_the_code.db ".mode csv" ".import /mnt/data/survey_results_public.csv survey_raw"
!sqlite3 b02_beyond_the_code.db ".mode csv" ".import /mnt/data/the-state-of-ai-2023-data.csv mck_raw"


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import pandas as pd
survey_df = pd.read_csv('survey_results_public.csv', low_memory=False)
mckinsey_df = pd.read_csv('the-state-of-ai-2023-data.csv', low_memory=False)
print("Survey dataset shape:", survey_df.shape)
print("McKinsey AI dataset shape:", mckinsey_df.shape)

In [ ]:
print("\nSurvey dataset columns:\n", survey_df.columns.tolist())
print("\nMcKinsey AI dataset columns:\n", mckinsey_df.columns.tolist())

In [ ]:
import duckdb

# Create in-memory SQL engine
conn = duckdb.connect(database=':memory:')

# Register DataFrames as tables for SQL querying
conn.register('survey', survey_df)
conn.register('mckinsey', mckinsey_df)

# Check what’s loaded
print(conn.execute("SHOW TABLES").fetchdf())
print(conn.execute("SELECT COUNT(*) AS survey_rows FROM survey").fetchdf())
print(conn.execute("SELECT COUNT(*) AS mckinsey_rows FROM mckinsey").fetchdf())

In [ ]:
query_ai_tools = """
SELECT "Country", "AIToolCurrently Using", COUNT(*) AS total_users
FROM survey
WHERE "AIToolCurrently Using" IS NOT NULL
GROUP BY "Country", "AIToolCurrently Using"
ORDER BY total_users DESC
LIMIT 10;
"""

ai_tools = conn.execute(query_ai_tools).fetchdf()
print(ai_tools)


In [ ]:
# Robust loader + DuckDB import for the problematic McKinsey file
import pandas as pd, duckdb, io, sys

mck_path = 'the-state-of-ai-2023-data.csv'
out_clean = 'the-state-of-ai-2023-data-clean.csv'

# Candidate encodings/delimiters to try
encodings = ['utf-8', 'utf-16', 'utf-16le', 'utf-16be', 'latin1', 'cp1252']
delims = [',', '\t', ';', '|']

def try_read(path, encoding, sep, nrows=5000):
    try:
        df = pd.read_csv(path, encoding=encoding, sep=sep, engine='c', nrows=nrows, low_memory=False)
        # quick sanity checks: enough columns and expected header names
        cols = [c.strip() for c in df.columns.astype(str).tolist()]
        if len(cols) <= 1:
            return None, f"Only 1 column with sep={sep}, enc={encoding}"
        # check for expected McKinsey columns somewhere
        expected = {'Year','Metric','Value','Category','Group','Exhibit'}
        if len(expected.intersection(set([c.lower() for c in cols]))) >= 1 or len(expected.intersection(set(cols))) >= 1:
            return df, f"OK with sep={sep}, enc={encoding}"
        # still accept if >2 columns
        return df, f"OK-ish with sep={sep}, enc={encoding}"
    except Exception as e:
        return None, f"FAIL sep={sep}, enc={encoding}, err={repr(e)}"

# Try combinations
found = False
lastmsg = []
for enc in encodings:
    for d in delims:
        df, msg = try_read(mck_path, enc, d)
        lastmsg.append(msg)
        if df is not None:
            print("SUCCESS:", msg)
            # read full file with the working params
            df_full = pd.read_csv(mck_path, encoding=enc, sep=d, engine='c', low_memory=False)
            # Normalize column names: strip whitespace
            df_full.columns = [c.strip() for c in df_full.columns.astype(str).tolist()]
            df_full.to_csv(out_clean, index=False, encoding='utf-8')
            print(f"Saved cleaned file to `{out_clean}` with encoding utf-8 and delimiter '{d}'.")
            found = True
            detected_enc = enc
            detected_delim = d
            break
    if found:
        break

# If not found, try python engine with sep=None to let pandas sniff
if not found:
    try:
        print("No clean read found with c-engine combos; trying python engine with sep=None (delimiter sniff).")
        df_try = pd.read_csv(mck_path, engine='python', sep=None, encoding='utf-8', low_memory=False)
        df_try.columns = [c.strip() for c in df_try.columns.astype(str).tolist()]
        df_try.to_csv(out_clean, index=False, encoding='utf-8')
        print(f"Saved cleaned file to `{out_clean}` using python engine sep=None (sniff).")
        found = True
        detected_enc = 'utf-8 (sniff)'
        detected_delim = 'sniffed'
    except Exception as e:
        last_err = e
        print("Python-engine sniff failed:", repr(e))

# As a last resort: try reading with latin1 and no delimiter (read whole as one column) then attempt to split rows by newline and commas
if not found:
    try:
        print("Last resort: reading with latin1 and then manual parsing.")
        raw = open(mck_path, 'rb').read()
        # try decode with latin1 then splitlines
        text = raw.decode('latin1', errors='replace')
        lines = text.splitlines()
        # Try to find a line with 'Year' or 'Metric' to locate header index
        header_idx = None
        for i, line in enumerate(lines[:30]):
            if 'Year' in line or 'Metric' in line:
                header_idx = i
                break
        if header_idx is None:
            header_idx = 0
        sample = "\n".join(lines[header_idx:header_idx+200])
        # attempt pandas read from the sample with sep ','
        df_manual = pd.read_csv(io.StringIO(sample), sep=',', engine='python', low_memory=False)
        df_manual.columns = [c.strip() for c in df_manual.columns.astype(str).tolist()]
        df_manual.to_csv(out_clean, index=False, encoding='utf-8')
        print(f"Saved cleaned file to `{out_clean}` using manual latin1 fallback.")
        found = True
        detected_enc = 'latin1_manual'
        detected_delim = ', (manual)'
    except Exception as e:
        print("Manual fallback failed:", repr(e))

if not found:
    print("FAILED to auto-detect a usable parse for the McKinsey file. Last messages:")
    for m in lastmsg[-6:]:
        print("  -", m)
    raise RuntimeError("Could not parse the McKinsey CSV automatically. Try opening it in Excel to inspect encoding/delimiter.")

# If we have a cleaned file, load into DuckDB
if found:
    print("\nNow importing cleaned CSV into DuckDB table `mck_clean`...")
    con = duckdb.connect(database=':memory:')
    # load survey_raw too if available in working dir
    try:
        con.execute("CREATE OR REPLACE TABLE mck_raw AS SELECT * FROM read_csv_auto('{}', header=True, all_varchar=True);".format(out_clean))
        print("✅ mck_raw table created in DuckDB from cleaned CSV.")
    except Exception as e:
        print("DuckDB import failed:", repr(e))
        raise

    # show sample
    df_sample = con.execute("SELECT * FROM mck_raw LIMIT 10").fetchdf()
    print("\nSample rows from mck_raw:")
    print(df_sample)
    # expose variables to user
    detected = {'detected_encoding': detected_enc, 'detected_delimiter': detected_delim, 'clean_path': out_clean}
    print("\nDETECTION:", detected)


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
%%bigquery
SELECT *
FROM `your_dataset.mck_clean`
LIMIT 10
